# Pythia-410m: 5M-token training

Requires **Google Colab Pro+**. **Runtime: A100 GPU, High-RAM** (Runtime → Change runtime type). Run `pythia410m_init.ipynb` first.

Distils each branch toward Pythia-2.8b for 5M tokens, with the gradient clipped at 1.0. Rows go to `MyDrive/gaugemla/grad_clip_5m/p410m_gc5m/runs/<stamp>/`. 

`BRANCH` picks one branch (`<method>_G<G>`; G = 4, 2, 1 is 25, 12.5, 6.25% of the KV cache); empty trains all 24, one after another. `TEST_TOKENS = 50000` is a quick test (about 7 optimiser steps) under its own stamp; 0 is the full run. A finished branch is skipped on a rerun.

Requires a Hugging Face token: add it as the Colab secret `HF_TOKEN` and give this notebook access.

In [ ]:
# @title Setup
BRANCH = "mla_w64_ours_unrope_cost_G4"  # @param ["mla_w64_ours_unrope_cost_G4", "decoupled_optwide_fisher_sigma_dkr64_G4", "pool_G4", "gauge3B_G4", "procrustes_simk_G4", "optimal_G4", "optwide_G4", "mla_ours_unrope_cost_G4", "decoupled_optwide_fisher_sigma_dkr64_G2", "pool_G2", "gauge3B_G2", "procrustes_simk_G2", "optimal_G2", "optwide_G2", "mla_w64_ours_unrope_cost_G2", "mla_ours_unrope_cost_G2", "decoupled_optwide_fisher_sigma_dkr64_G1", "pool_G1", "gauge3B_G1", "procrustes_simk_G1", "optimal_G1", "mla_w64_ours_unrope_cost_G1", "mla_ours_unrope_cost_G1", "mha_G0", ""] {allow-input: true}
TEST_TOKENS = 0  # @param {type:"integer"}

import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
# The repository is public. While it is private, uncomment these lines and add the Colab secret READ_MLA_TOKEN
# (a GitHub token that can read it):
# from google.colab import userdata
# os.environ["READ_MLA_TOKEN"] = userdata.get("READ_MLA_TOKEN")
# with open("/content/askpass.sh", "w") as f:
#     f.write('#!/bin/sh\necho "$READ_MLA_TOKEN"\n')
# os.chmod("/content/askpass.sh", 0o700)
# os.environ["GIT_ASKPASS"] = "/content/askpass.sh"
os.environ["GIT_TERMINAL_PROMPT"] = "0"
if os.path.isdir("/content/repo/.git"):
    subprocess.run(["git", "-C", "/content/repo", "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", "/content/repo", "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                    "https://github.com/anon5159753/gaugemla.git", "/content/repo"], check=True)
    subprocess.run(["git", "-C", "/content/repo", "sparse-checkout", "set", "src", "colab"], check=True)
sys.path.insert(0, "/content/repo/colab")
import launch

In [ ]:
# @title 5M training (p410m_gc5m)
launch.run("p410m_gc5m", "run", BRANCH=BRANCH, TEST_TOKENS=TEST_TOKENS)

In [ ]:
# @title KL to the teacher at t = 0 and at the end of training, beside results/
launch.kl_table("p410m_gc5m", TEST_TOKENS)